# Pac-Man — Monte Carlo Comparison of All Agents

This notebook drives every agent we have against the canonical Pac-Man maze
(`HighScorePacmanEnv`) for `N_EPISODES` Monte Carlo episodes each, then
compares them across:

- **Score** (peak score in episode — robust to death-policy resets)
- **Level-clear rate**
- **Time-to-failure** (steps before the first death)
- **Hazard exposure** (cumulative count of steps with a non-scared ghost within Manhattan distance 2)
- **Deaths**

Agents covered:

- Search baselines: `astar_static`, `astar_replan`, `astar_risk` (λ ∈ {0.5, 1, 2, 5})
- Hand-coded heuristic baseline (`heuristic`)
- Random baseline (`random`)
- Q-learning models 1, 2, 3 (three death-penalty variants from `RL/models/`)

Outputs are written to `Agents/eval/results/` (per-episode CSVs and
`summary.json`). Re-running with the same `N_EPISODES` reuses cached results
from disk.

In [1]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def _find_repo_root() -> Path:
    candidates = []
    nb_env = os.environ.get("MC_REPO_ROOT")
    if nb_env:
        candidates.append(Path(nb_env))
    candidates.append(Path.cwd())
    candidates.append(Path.cwd().parent)
    for c in list(candidates):
        cur = c.resolve()
        for _ in range(6):
            if (cur / "Agents" / "eval" / "monte_carlo.py").exists():
                return cur
            if cur == cur.parent:
                break
            cur = cur.parent
    raise RuntimeError("could not locate repo root containing Agents/eval/")


REPO_ROOT = _find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"REPO_ROOT = {REPO_ROOT}")

from Agents.eval.monte_carlo import run_many, write_csv  # noqa: E402
from Agents.eval.agents import AGENT_FACTORIES  # noqa: E402

RESULTS_DIR = REPO_ROOT / "Agents" / "eval" / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"results -> {RESULTS_DIR}")
print(f"available agents: {list(AGENT_FACTORIES)}")

REPO_ROOT = /Users/michael.liu/Documents/School/CS520/cs520-project
results -> /Users/michael.liu/Documents/School/CS520/cs520-project/Agents/eval/results
available agents: ['astar_static', 'astar_replan', 'astar_risk_l1', 'astar_risk_l05', 'astar_risk_l2', 'astar_risk_l5', 'heuristic', 'random', 'qlearning_model1', 'qlearning_model2', 'qlearning_model3']


## Configuration

Tune these knobs to trade speed vs. statistical confidence. The risk-aware
A* agents are the slowest (~0.7s/episode); everything else is sub-100ms.

In [2]:
N_EPISODES = 50
BASE_SEED = 1000
DEATH_POLICY = "model2"
USE_CACHE = True

MAIN_AGENTS = [
    "random",
    "heuristic",
    "astar_static",
    "astar_replan",
    "astar_risk_l1",
    "qlearning_model1",
    "qlearning_model2",
    "qlearning_model3",
]

LAMBDA_AGENTS = [
    ("astar_risk_l05", 0.5),
    ("astar_risk_l1", 1.0),
    ("astar_risk_l2", 2.0),
    ("astar_risk_l5", 5.0),
]

PALETTE = {
    "random": "#9e9e9e",
    "heuristic": "#ffb300",
    "astar_static": "#1e88e5",
    "astar_replan": "#43a047",
    "astar_risk_l1": "#8e24aa",
    "astar_risk_l05": "#ce93d8",
    "astar_risk_l2": "#6a1b9a",
    "astar_risk_l5": "#4a148c",
    "qlearning_model1": "#e53935",
    "qlearning_model2": "#fb8c00",
    "qlearning_model3": "#6d4c41",
}

print(f"Will run {N_EPISODES} episodes per agent on '{DEATH_POLICY}' env.")
print(f"  main comparison: {len(MAIN_AGENTS)} agents")
print(f"  lambda sweep:    {len(LAMBDA_AGENTS)} settings")

Will run 50 episodes per agent on 'model2' env.
  main comparison: 8 agents
  lambda sweep:    4 settings


## Run experiments (with disk cache)

A `(agent, n_episodes, seed, death_policy)` tuple keys each cached CSV. Delete
the file or set `USE_CACHE = False` above to force a re-run.

In [ ]:
def cache_path(agent: str) -> Path:
    return RESULTS_DIR / f"{agent}__n{N_EPISODES}_seed{BASE_SEED}_{DEATH_POLICY}.csv"


def load_or_run(agent: str) -> pd.DataFrame:
    path = cache_path(agent)
    if USE_CACHE and path.exists():
        return pd.read_csv(path)
    summary = run_many(
        agent,
        n_episodes=N_EPISODES,
        base_seed=BASE_SEED,
        death_policy=DEATH_POLICY,
        progress=lambda msg: print(msg),
    )
    write_csv(summary.raw, path)
    print(
        f"  done {agent:20s}  avg_score={summary.avg_score:7.1f}  "
        f"clear={summary.clear_rate:.2f}  ttf={summary.avg_time_to_failure:6.1f}  "
        f"wall={summary.wall_time_s:.1f}s"
    )
    return pd.read_csv(path)


frames = []
for agent in MAIN_AGENTS:
    print(f"--> {agent}")
    frames.append(load_or_run(agent))

extra_lambda_agents = [name for (name, _) in LAMBDA_AGENTS if name not in MAIN_AGENTS]
for agent in extra_lambda_agents:
    print(f"--> {agent}  (lambda sweep)")
    frames.append(load_or_run(agent))

raw = pd.concat(frames, ignore_index=True)
print(f"\nTotal episodes: {len(raw)} across {raw['agent'].nunique()} agents")
raw.head()

--> random
  random: 10/50 eps  avg_score=112.0


## Summary table

In [ ]:
def summarize(df: pd.DataFrame) -> pd.DataFrame:
    grouped = df.groupby("agent")
    summary = pd.DataFrame({
        "episodes": grouped.size(),
        "score_mean": grouped["score"].mean(),
        "score_std": grouped["score"].std(ddof=0),
        "score_median": grouped["score"].median(),
        "score_max": grouped["score"].max(),
        "clear_rate": grouped["cleared"].mean(),
        "avg_deaths": grouped["deaths"].mean(),
        "avg_steps": grouped["steps"].mean(),
        "avg_ttf": grouped["time_to_failure"].mean(),
        "avg_hazard": grouped["hazard_exposure"].mean(),
    }).round(2)
    return summary.sort_values("score_mean", ascending=False)


summary_df = summarize(raw)
summary_path = RESULTS_DIR / f"summary_n{N_EPISODES}_seed{BASE_SEED}.json"
summary_path.write_text(summary_df.to_json(orient="index", indent=2), encoding="utf-8")
print(f"summary -> {summary_path}")
summary_df

## Score comparison

Mean peak score per episode, with 1σ error bars. Black diamonds are the
single best score observed for the agent across the run.

In [ ]:
def colors_for(agents):
    return [PALETTE.get(a, "#607d8b") for a in agents]


main_summary = summary_df.loc[[a for a in MAIN_AGENTS if a in summary_df.index]]
ordered = main_summary.sort_values("score_mean", ascending=True)

fig, ax = plt.subplots(figsize=(10, 5))
ypos = np.arange(len(ordered))
ax.barh(
    ypos,
    ordered["score_mean"],
    xerr=ordered["score_std"],
    color=colors_for(ordered.index),
    edgecolor="black",
    linewidth=0.5,
    capsize=4,
)
ax.scatter(
    ordered["score_max"],
    ypos,
    marker="D",
    color="black",
    s=24,
    label="best of run",
    zorder=3,
)
ax.set_yticks(ypos)
ax.set_yticklabels(ordered.index)
ax.set_xlabel("Score (peak)")
ax.set_title(f"Mean peak score per episode (n={N_EPISODES} each)")
ax.legend(loc="lower right")
ax.grid(axis="x", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()

## Score distribution (boxplot)

In [ ]:
main_raw = raw[raw["agent"].isin(MAIN_AGENTS)].copy()
order = main_summary.sort_values("score_mean", ascending=False).index.tolist()
groups = [main_raw.loc[main_raw["agent"] == a, "score"].values for a in order]

fig, ax = plt.subplots(figsize=(11, 5))
bp = ax.boxplot(
    groups,
    labels=order,
    patch_artist=True,
    showfliers=True,
    widths=0.6,
)
for patch, agent in zip(bp["boxes"], order):
    patch.set_facecolor(PALETTE.get(agent, "#607d8b"))
    patch.set_alpha(0.85)
for median in bp["medians"]:
    median.set_color("black")
    median.set_linewidth(1.5)
ax.set_ylabel("Score (peak)")
ax.set_title(f"Score distribution per agent  (n={N_EPISODES})")
plt.xticks(rotation=20, ha="right")
ax.grid(axis="y", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()

## Survival metrics — clear-rate, deaths, time-to-failure

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

ord_clear = main_summary.sort_values("clear_rate", ascending=True)
axes[0].barh(
    ord_clear.index,
    ord_clear["clear_rate"],
    color=colors_for(ord_clear.index),
    edgecolor="black",
    linewidth=0.5,
)
axes[0].set_xlabel("Level-clear rate")
axes[0].set_xlim(0, 1)
axes[0].set_title("Level clears (higher = better)")
axes[0].grid(axis="x", linestyle=":", alpha=0.5)

ord_d = main_summary.sort_values("avg_deaths", ascending=False)
axes[1].barh(
    ord_d.index,
    ord_d["avg_deaths"],
    color=colors_for(ord_d.index),
    edgecolor="black",
    linewidth=0.5,
)
axes[1].set_xlabel("Average deaths per episode")
axes[1].set_xlim(0, 3.05)
axes[1].set_title("Deaths (lower = better)")
axes[1].grid(axis="x", linestyle=":", alpha=0.5)

ord_ttf = main_summary.sort_values("avg_ttf", ascending=True)
axes[2].barh(
    ord_ttf.index,
    ord_ttf["avg_ttf"],
    color=colors_for(ord_ttf.index),
    edgecolor="black",
    linewidth=0.5,
)
axes[2].set_xlabel("Steps before first death")
axes[2].set_title("Time-to-failure (higher = better)")
axes[2].grid(axis="x", linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()

## Time-to-failure CDF

How quickly does each agent meet its first ghost? The CDF tells us at step
*x*, what fraction of episodes for that agent had already lost a life. Curves
that stay low/right are safer.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5.5))
for agent in MAIN_AGENTS:
    values = np.sort(raw.loc[raw["agent"] == agent, "time_to_failure"].values)
    if len(values) == 0:
        continue
    cdf = np.arange(1, len(values) + 1) / len(values)
    ax.step(
        values,
        cdf,
        where="post",
        color=PALETTE.get(agent, "#607d8b"),
        label=agent,
        linewidth=2,
    )
ax.set_xlabel("Steps before first death")
ax.set_ylabel("Cumulative fraction of episodes")
ax.set_title("Time-to-failure CDF (right = safer)")
ax.set_ylim(0, 1.02)
ax.grid(linestyle=":", alpha=0.5)
ax.legend(loc="lower right", ncol=2, frameon=True)
plt.tight_layout()
plt.show()

## Risk vs reward — score vs hazard exposure

Cumulative hazard exposure on the x-axis (the more time spent next to a
non-scared ghost, the further right). Score on the y-axis. The Pareto-good
agents are top-left (high score, low hazard).

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for agent in MAIN_AGENTS:
    sub = raw[raw["agent"] == agent]
    ax.scatter(
        sub["hazard_exposure"],
        sub["score"],
        s=28,
        color=PALETTE.get(agent, "#607d8b"),
        alpha=0.55,
        edgecolor="white",
        linewidth=0.5,
    )
    centroid = (sub["hazard_exposure"].mean(), sub["score"].mean())
    ax.scatter(
        [centroid[0]],
        [centroid[1]],
        marker="X",
        s=180,
        color=PALETTE.get(agent, "#607d8b"),
        edgecolor="black",
        linewidth=1.2,
        label=agent,
        zorder=5,
    )
ax.set_xlabel("Hazard exposure (cumulative steps within ghost radius 2)")
ax.set_ylabel("Score (peak)")
ax.set_title("Risk vs reward (X = mean per agent)")
ax.grid(linestyle=":", alpha=0.5)
ax.legend(loc="best", ncol=2, frameon=True)
plt.tight_layout()
plt.show()

## Lambda sweep — Risk-Aware A*

Trade-off between greedy A* (`λ=0`) and over-cautious risk avoidance
(`λ=5`). The decision-theoretic search formula is `f(n) = g(n) + h(n) + λ ·
E[risk(n)]`.

In [ ]:
lambda_rows = []
for agent_name, lam in LAMBDA_AGENTS:
    sub = raw[raw["agent"] == agent_name]
    if sub.empty:
        continue
    lambda_rows.append({
        "lambda": lam,
        "agent": agent_name,
        "score_mean": sub["score"].mean(),
        "score_std": sub["score"].std(ddof=0),
        "clear_rate": sub["cleared"].mean(),
        "avg_deaths": sub["deaths"].mean(),
        "avg_ttf": sub["time_to_failure"].mean(),
        "avg_hazard": sub["hazard_exposure"].mean(),
    })
lambda_df = pd.DataFrame(lambda_rows).sort_values("lambda")
display(lambda_df)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].errorbar(
    lambda_df["lambda"],
    lambda_df["score_mean"],
    yerr=lambda_df["score_std"],
    marker="o",
    color="#8e24aa",
    capsize=4,
    linewidth=2,
)
axes[0].set_xlabel("λ")
axes[0].set_ylabel("Score (peak)")
axes[0].set_title("Score vs λ")
axes[0].grid(linestyle=":", alpha=0.5)

axes[1].plot(
    lambda_df["lambda"],
    lambda_df["avg_ttf"],
    marker="o",
    color="#1e88e5",
    linewidth=2,
    label="time-to-failure",
)
axes[1].plot(
    lambda_df["lambda"],
    lambda_df["avg_hazard"],
    marker="s",
    color="#e53935",
    linewidth=2,
    label="hazard exposure",
)
axes[1].set_xlabel("λ")
axes[1].set_title("Survival vs λ")
axes[1].grid(linestyle=":", alpha=0.5)
axes[1].legend()

axes[2].plot(
    lambda_df["lambda"],
    lambda_df["clear_rate"],
    marker="o",
    color="#43a047",
    linewidth=2,
)
axes[2].set_xlabel("λ")
axes[2].set_ylabel("Level-clear rate")
axes[2].set_title("Clear rate vs λ")
axes[2].set_ylim(0, 1)
axes[2].grid(linestyle=":", alpha=0.5)

plt.tight_layout()
plt.show()

## Q-learning death-policy ablation

Each Q-learning model was trained with a different death-penalty scheme
(model 1: keep score on death; model 2: zero on death; model 3: scaled
penalty). The ablation shows what that does at evaluation time.

In [ ]:
ql_agents = [a for a in MAIN_AGENTS if a.startswith("qlearning")]
ql_summary = summary_df.loc[ql_agents][[
    "score_mean", "score_std", "clear_rate", "avg_deaths", "avg_ttf", "avg_hazard"
]]
display(ql_summary)

metrics = ["score_mean", "clear_rate", "avg_deaths", "avg_ttf"]
titles = ["Mean score", "Clear rate", "Avg deaths", "Avg time-to-failure"]
fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, metric, title in zip(axes, metrics, titles):
    bars = ax.bar(
        ql_summary.index,
        ql_summary[metric],
        color=[PALETTE[a] for a in ql_summary.index],
        edgecolor="black",
        linewidth=0.5,
    )
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=15)
    ax.grid(axis="y", linestyle=":", alpha=0.5)
plt.tight_layout()
plt.show()

## Per-episode score traces

Useful for spotting variance and lucky/unlucky seeds. Each line is one
agent across the seed range.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))
for agent in MAIN_AGENTS:
    sub = raw[raw["agent"] == agent].sort_values("episode")
    ax.plot(
        sub["episode"],
        sub["score"],
        marker="o",
        markersize=3,
        linewidth=1.2,
        alpha=0.85,
        color=PALETTE.get(agent, "#607d8b"),
        label=agent,
    )
ax.set_xlabel("Episode index")
ax.set_ylabel("Score (peak)")
ax.set_title(f"Per-episode score traces (n={N_EPISODES})")
ax.grid(linestyle=":", alpha=0.5)
ax.legend(loc="best", ncol=2, frameon=True)
plt.tight_layout()
plt.show()

## Head-to-head heatmap

Pairwise: how often does agent A beat agent B on the same seed? Diagonal is
50% by definition.

In [ ]:
pivot = raw.pivot_table(index="seed", columns="agent", values="score", aggfunc="first")
agents_present = [a for a in MAIN_AGENTS if a in pivot.columns]
pivot = pivot[agents_present]

n = len(agents_present)
matrix = np.full((n, n), 0.5)
for i, a in enumerate(agents_present):
    for j, b in enumerate(agents_present):
        if i == j:
            continue
        diffs = pivot[a].values - pivot[b].values
        wins = np.sum(diffs > 0)
        ties = np.sum(diffs == 0)
        total = len(diffs)
        if total > 0:
            matrix[i, j] = (wins + 0.5 * ties) / total

fig, ax = plt.subplots(figsize=(8.5, 7))
im = ax.imshow(matrix, cmap="RdYlGn", vmin=0, vmax=1)
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(agents_present, rotation=35, ha="right")
ax.set_yticklabels(agents_present)
for i in range(n):
    for j in range(n):
        text_color = "white" if abs(matrix[i, j] - 0.5) > 0.3 else "black"
        ax.text(j, i, f"{matrix[i, j]:.2f}", ha="center", va="center", color=text_color, fontsize=9)
ax.set_title("P(row beats column on same seed)")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()

## Takeaways

- **Best raw scorer**: see the score-comparison bars above — typically
  `astar_replan` or `astar_risk_l1`, both of which use ghost-aware planning.
- **Most reliable clears**: whichever agent dominates the clear-rate plot.
  The static A* often clears under easy seeds but dies more on adversarial
  seeds.
- **Safest agent (highest TTF, lowest hazard)**: usually a high-λ
  `astar_risk_*` variant — bought with lower throughput.
- **Q-learning trade-off**: model 1 (keep-score-on-death) tends to learn a
  greedy pellet-grabbing policy; model 3 (heavy scaled penalty) is the most
  cautious; model 2 sits in between.

Adjust `N_EPISODES` upward and re-run for tighter error bars before quoting
numbers in the report.